In [ ]:
import json, os, shutil, subprocess, sys, time, uuid, requests, torch
from pathlib import Path

assert torch.cuda.is_available(), 'GPU is required for Krea 2 Turbo inference on Colab.'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
COMFY = Path('/content/ComfyUI')
MODELS = COMFY / 'models'
CUSTOM_NODES = COMFY / 'custom_nodes'
SETUP_MARKER = Path('/content/.h3_comfyui_setup_v1')

if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt'), 'huggingface_hub', 'hf_xet', 'hf_transfer', 'pillow', 'requests'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'hf_transfer', 'pillow', 'requests'], check=False)

MODE = os.environ.get('KREA2_MODE', 'text2image').strip().lower()
KREA_EDIT_DIR = CUSTOM_NODES / 'comfyui-krea2edit'
need_comfy_restart = False
if MODE == 'edit' and not KREA_EDIT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/lbouaraba/comfyui-krea2edit.git', str(KREA_EDIT_DIR)], check=True)
    need_comfy_restart = True

# Auto-Idle Watchdog: self-terminate after 15 mins of inactivity to save CU
_ACTIVITY_FILE = Path('/content/.h3_last_activity')
_ACTIVITY_FILE.write_text(str(time.time()))
if not Path('/content/.h3_watchdog_started').exists():
    Path('/content/.h3_watchdog_started').touch()
    subprocess.Popen([
        sys.executable, '-c',
        "import os, time, pathlib, requests\n"
        "act = pathlib.Path('/content/.h3_last_activity')\n"
        "while True:\n"
        "    time.sleep(30)\n"
        "    try:\n"
        "        q = requests.get('http://127.0.0.1:8188/queue', timeout=3).json()\n"
        "        if q.get('queue_running') or q.get('queue_pending'):\n"
        "            act.write_text(str(time.time()))\n"
        "            continue\n"
        "    except Exception:\n"
        "        pass\n"
        "    last = float(act.read_text()) if act.exists() else time.time()\n"
        "    if time.time() - last > 900:\n"
        "        os.system('kill -9 1')\n"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)

from huggingface_hub import hf_hub_download

def ensure_hf_file(repo_id: str, remote_file: str, dest_path: Path, label: str):
    if dest_path.is_file() and dest_path.stat().st_size > 1024:
        return
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'[Krea2 Colab] Downloading {label} ({repo_id}/{remote_file})...', flush=True)
    p_rem = Path(remote_file)
    dl = Path(hf_hub_download(
        repo_id=repo_id,
        filename=p_rem.name,
        subfolder=None if str(p_rem.parent) in ('', '.') else str(p_rem.parent).replace('\\', '/'),
        local_dir=str(dest_path.parent),
    ))
    if dl.resolve() != dest_path.resolve():
        shutil.move(str(dl), str(dest_path))
    print(f'[Krea2 Colab] Ready: {dest_path.name}', flush=True)

ensure_hf_file('Comfy-Org/Krea-2', 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors', MODELS / 'text_encoders' / 'qwen3vl_4b_fp8_scaled.safetensors', 'Qwen3-VL 4B Text Encoder')
ensure_hf_file('Comfy-Org/Krea-2', 'vae/qwen_image_vae.safetensors', MODELS / 'vae' / 'qwen_image_vae.safetensors', 'Qwen Image VAE')

CKPT_ID = os.environ.get('KREA2_CHECKPOINT', 'gonzalomoKrea2_v40.safetensors').strip()
CKPT_MAP = {
    'gonzalomoKrea2_v40.safetensors': ('metaloz/imgstuff', 'gonzalomoKrea2_v40.safetensors', 'gonzalomoKrea2_v40.safetensors'),
    'pornmasterKrea2_v2TurboInt8.safetensors': ('mpasila/Krea-2-Models', 'pornmasterKrea2_v2TurboInt8.safetensors', 'pornmasterKrea2_v2TurboInt8.safetensors'),
    'museByStableYogi_v25EXTENDEDTURBO.safetensors': ('mpasila/Krea-2-Models', 'museByStableYogi_v25EXTENDEDTURBO.safetensors', 'museByStableYogi_v25EXTENDEDTURBO.safetensors'),
    'comfy_org_krea2_turbo_fp8': ('Comfy-Org/Krea-2', 'diffusion_models/krea2_turbo_fp8_scaled.safetensors', 'krea2_turbo_fp8_scaled.safetensors'),
}
ckpt_repo, ckpt_rem, ckpt_local = CKPT_MAP.get(CKPT_ID, CKPT_MAP['gonzalomoKrea2_v40.safetensors'])
ensure_hf_file(ckpt_repo, ckpt_rem, MODELS / 'diffusion_models' / ckpt_local, f'Krea 2 Checkpoint ({ckpt_local})')

LORA_REPO = os.environ.get('KREA2_LORA_REPO', '').strip()
LORA_FILE = os.environ.get('KREA2_LORA_FILE', '').strip()
LORA_STRENGTH = float(os.environ.get('KREA2_LORA_STRENGTH', '0.85'))
lora_local_name = ''
if LORA_REPO and LORA_FILE:
    lora_local_name = Path(LORA_FILE).name
    ensure_hf_file(LORA_REPO, LORA_FILE, MODELS / 'loras' / lora_local_name, f'Style LoRA ({lora_local_name})')

if MODE == 'edit':
    ensure_hf_file('conradlocke/krea2-identity-edit', 'krea2_identity_edit_v1_2.safetensors', MODELS / 'loras' / 'krea' / 'krea2_identity_edit_v1_2.safetensors', 'Krea 2 Identity Edit Adapter')

PORT = 8188
BASE_URL = f'http://127.0.0.1:{PORT}'
LOG_PATH = Path('/content/comfyui_krea2.log')

if need_comfy_restart:
    subprocess.run(['pkill', '-f', 'ComfyUI/main.py'], check=False)
    time.sleep(2)

try:
    requests.get(BASE_URL + '/system_stats', timeout=2).raise_for_status()
    print('[Krea2 Colab] Reusing active ComfyUI server on port 8188.')
except Exception:
    print('[Krea2 Colab] Starting ComfyUI server on port 8188...', flush=True)
    log_f = open(LOG_PATH, 'a')
    launch = [sys.executable, str(COMFY / 'main.py'), '--listen', '127.0.0.1', '--port', str(PORT), '--disable-auto-launch']
    if torch.cuda.get_device_properties(0).total_memory < 22 * 2**30:
        launch.append('--lowvram')
    proc = subprocess.Popen(launch, cwd=str(COMFY), stdout=log_f, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(120):
        if proc.poll() is not None:
            raise RuntimeError('ComfyUI exited early:\n' + (LOG_PATH.read_text()[-3000:] if LOG_PATH.is_file() else ''))
        try:
            requests.get(BASE_URL + '/system_stats', timeout=2).raise_for_status()
            break
        except Exception:
            time.sleep(1.5)
    else:
        raise TimeoutError('ComfyUI startup timed out.')

PROMPT_FILE = Path(os.environ.get('KREA2_PROMPT_FILE', '/content/krea2_prompt.txt'))
PROMPT_TEXT = PROMPT_FILE.read_text(encoding='utf-8').strip() if PROMPT_FILE.is_file() else os.environ.get('KREA2_PROMPT', '')
WIDTH = int(os.environ.get('KREA2_WIDTH', '1024'))
HEIGHT = int(os.environ.get('KREA2_HEIGHT', '1024'))
STEPS = max(4, min(28, int(os.environ.get('KREA2_STEPS', '8'))))
CFG = max(1.0, min(7.0, float(os.environ.get('KREA2_CFG', '1.0'))))
SEED = int(os.environ.get('KREA2_SEED', '42'))
OUT_PATH = Path(os.environ.get('KREA2_OUTPUT_PATH', '/content/krea2_output.png'))
REF1_PATH = os.environ.get('KREA2_REF1', '').strip()
REF2_PATH = os.environ.get('KREA2_REF2', '').strip()

input_dir = COMFY / 'input'
input_dir.mkdir(parents=True, exist_ok=True)

if MODE == 'edit' and REF1_PATH and Path(REF1_PATH).is_file():
    ref1_name = f'krea2_ref1_{uuid.uuid4().hex[:8]}.png'
    shutil.copy2(REF1_PATH, input_dir / ref1_name)
    workflow = {
        '1': {'class_type': 'LoadImage', 'inputs': {'image': ref1_name}},
        '3': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'krea2', 'device': 'default'}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_vae.safetensors'}},
        '5': {'class_type': 'UNETLoader', 'inputs': {'unet_name': ckpt_local, 'weight_dtype': 'default'}},
        '6': {'class_type': 'LoraLoaderModelOnly', 'inputs': {'model': ['5', 0], 'lora_name': 'krea/krea2_identity_edit_v1_2.safetensors', 'strength_model': 1.0}},
        '7': {'class_type': 'VAEEncode', 'inputs': {'pixels': ['1', 0], 'vae': ['4', 0]}},
        '8': {'class_type': 'EmptySD3LatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
        '9': {'class_type': 'Krea2EditModelPatch', 'inputs': {'model': ['6', 0], 'source_latent': ['7', 0], 'vae': ['4', 0], 'source_image': ['1', 0], 'target_latent': ['8', 0], 'ref_boost': 1.0, 'ref_boost_a': 1.0, 'fit_mode': 'fit'}},
        '10': {'class_type': 'Krea2EditGroundedEncode', 'inputs': {'clip': ['3', 0], 'image': ['1', 0], 'prompt': PROMPT_TEXT, 'grounding_px': 768}},
        '11': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['10', 0]}},
        '12': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'model': ['9', 0], 'shift': 4.0}},
        '13': {'class_type': 'KSampler', 'inputs': {'model': ['12', 0], 'positive': ['10', 0], 'negative': ['11', 0], 'latent_image': ['8', 0], 'seed': SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'euler', 'scheduler': 'beta', 'denoise': 1.0}},
        '14': {'class_type': 'VAEDecode', 'inputs': {'samples': ['13', 0], 'vae': ['4', 0]}},
        '15': {'class_type': 'SaveImage', 'inputs': {'images': ['14', 0], 'filename_prefix': 'krea2_colab'}},
    }
    if REF2_PATH and Path(REF2_PATH).is_file():
        ref2_name = f'krea2_ref2_{uuid.uuid4().hex[:8]}.png'
        shutil.copy2(REF2_PATH, input_dir / ref2_name)
        workflow['2'] = {'class_type': 'LoadImage', 'inputs': {'image': ref2_name}}
        workflow['16'] = {'class_type': 'VAEEncode', 'inputs': {'pixels': ['2', 0], 'vae': ['4', 0]}}
        workflow['9']['inputs']['source_latent_b'] = ['16', 0]
        workflow['9']['inputs']['source_image_b'] = ['2', 0]
        workflow['10']['inputs']['image_b'] = ['2', 0]
    save_node_id = '15'
else:
    model_src = ['1', 0]
    clip_src = ['2', 0]
    workflow = {
        '1': {'class_type': 'UNETLoader', 'inputs': {'unet_name': ckpt_local, 'weight_dtype': 'default'}},
        '2': {'class_type': 'CLIPLoader', 'inputs': {'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'krea2', 'device': 'default'}},
        '3': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'qwen_image_vae.safetensors'}},
    }
    if lora_local_name:
        workflow['1_lora'] = {
            'class_type': 'LoraLoader',
            'inputs': {'model': model_src, 'clip': clip_src, 'lora_name': lora_local_name, 'strength_model': LORA_STRENGTH, 'strength_clip': LORA_STRENGTH}
        }
        model_src = ['1_lora', 0]
        clip_src = ['1_lora', 1]
    workflow.update({
        '4': {'class_type': 'ModelSamplingAuraFlow', 'inputs': {'model': model_src, 'shift': 4.0}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'clip': clip_src, 'text': PROMPT_TEXT}},
        '6': {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['5', 0]}},
        '7': {'class_type': 'EmptyLatentImage', 'inputs': {'width': WIDTH, 'height': HEIGHT, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {'model': ['4', 0], 'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0], 'seed': SEED, 'steps': STEPS, 'cfg': CFG, 'sampler_name': 'euler', 'scheduler': 'beta', 'denoise': 1.0}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['3', 0]}},
        '10': {'class_type': 'SaveImage', 'inputs': {'images': ['9', 0], 'filename_prefix': 'krea2_colab'}},
    })
    save_node_id = '10'

res = requests.post(BASE_URL + '/prompt', json={'prompt': workflow, 'client_id': 'krea2-colab'}, timeout=30)
if not res.ok:
    raise RuntimeError(f'ComfyUI prompt error {res.status_code}: {res.text}')
prompt_id = res.json()['prompt_id']
print(f'[Krea2 Colab] Rendering image ({WIDTH}x{HEIGHT}, {STEPS} steps) with {ckpt_local}... prompt_id={prompt_id}', flush=True)

t0 = time.time()
while time.time() - t0 < 300:
    hist = requests.get(BASE_URL + f'/history/{prompt_id}', timeout=15).json()
    if prompt_id in hist:
        run = hist[prompt_id]
        if run.get('status', {}).get('status_str') == 'error':
            raise RuntimeError('ComfyUI execution error: ' + json.dumps(run.get('status', {})))
        outputs = run.get('outputs', {})
        if save_node_id in outputs:
            imgs = outputs[save_node_id].get('images', [])
            if imgs:
                img_info = imgs[0]
                src_png = COMFY / 'output' / img_info.get('subfolder', '') / img_info['filename']
                OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_png, OUT_PATH)
                _ACTIVITY_FILE.write_text(str(time.time()))
                print(f'[Krea2 Colab] Completed in {round(time.time() - t0, 1)}s -> {OUT_PATH} ({OUT_PATH.stat().st_size} bytes)', flush=True)
                break
    time.sleep(0.8)
else:
    raise TimeoutError('Krea 2 Colab image generation timed out.')


GPU: Tesla T4 | VRAM GiB: 14.6


[Krea2 Colab] Downloading Qwen3-VL 4B Text Encoder (Comfy-Org/Krea-2/text_encoders/qwen3vl_4b_fp8_scaled.safetensors)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


text_encoders/qwen3vl_4b_fp8_scaled.safe(…): reconstructing file:   0%|          |  0.00B / 5.24GB            

...

gonzalomoKrea2_v40.safetensors: downloading bytes:           |  0.00B            

[Krea2 Colab] Ready: gonzalomoKrea2_v40.safetensors


[Krea2 Colab] Starting ComfyUI server on port 8188...


[Krea2 Colab] Rendering image (1344x768, 20 steps) with gonzalomoKrea2_v40.safetensors... prompt_id=8520abd2-cefd-48f7-b3b7-fad0e29c86ba


[Krea2 Colab] Completed in 212.0s -> /content/krea2_out_ac1fd7b2.png (1489628 bytes)
